# Mô hình C: fine-tune BARTpho-syllable (Kaggle / Colab)

Notebook mỏng: mọi logic nằm trong `src/vspell/models/bartpho.py` và `scripts/`. Chạy lần lượt từ trên xuống.

**Chuẩn bị (làm một lần trên máy):**
1. Repo đã push lên GitHub (điền `REPO_URL` ở ô dưới; repo private thì dùng URL có token, hoặc upload repo thành dataset).
2. **Upload dữ liệu thành Kaggle dataset**: tạo dataset mới (vd tên `vspell-data`), upload các file
   `data/processed/*.jsonl` (`syn_train.jsonl`, `syn_val.jsonl`, `syn_test.jsonl`, `vsec_test.jsonl`, `viwiki_test.jsonl`;
   ablation thêm `syn_train_d05/d12/d25.jsonl`). Dữ liệu không nằm trong git.
   Trong notebook: *Add input* → chọn dataset đó → nó xuất hiện ở `/kaggle/input/vspell-data/`.
   Colab: đặt các file đó lên Google Drive và sửa `DATA_SRC`.
3. Kaggle: *Settings → Accelerator → GPU T4 x2* (2 GPU thì code tự giảm tích lũy gradient để batch hiệu dụng vẫn 32) hoặc P100, *Internet: On* (để tải `vinai/bartpho-syllable`).

**Thời gian ước lượng (fp16):** 150k cặp × 3 epoch ≈ 14k bước; 1 x T4 khoảng 3–4,5 giờ, T4 x2 khoảng 2–3 giờ; P100 (không tăng tốc fp16 nhiều) khoảng 4–5 giờ + ~20–40 phút predict 3 tập test.
Kaggle giới hạn 12 giờ/phiên; checkpoint lưu mỗi 1000 bước, nếu bị ngắt thì chạy lại ô train với `--resume`.

In [ ]:
# ==== Sửa các biến này ====
REPO_URL = "https://github.com/<user>/<repo>.git"   # URL repo GitHub
BRANCH   = "main"
DATA_SRC = "/kaggle/input/vspell-data"              # thư mục chứa *.jsonl (Colab: "/content/drive/MyDrive/vspell-data")
RUN_ABLATION = False                                # True: train thêm 3 mô hình mật độ nhiễu (50k cặp, 1 epoch)

In [ ]:
import os, subprocess
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
REPO = f"{WORK}/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
!git log --oneline -1

In [ ]:
# Kaggle/Colab đã có torch + CUDA: chỉ cài phần còn thiếu, KHÔNG cài lại torch theo requirements.txt
!pip install -q "transformers>=4.41" "datasets>=2.19" "accelerate>=0.30" sentencepiece PyYAML
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Colab + Drive: bỏ comment 2 dòng dưới
# from google.colab import drive; drive.mount("/content/drive")
import glob, shutil
os.makedirs("data/processed", exist_ok=True)
for f in glob.glob(f"{DATA_SRC}/**/*.jsonl", recursive=True):
    dst = f"data/processed/{os.path.basename(f)}"
    if not os.path.exists(dst):
        os.symlink(f, dst)
!ls -la data/processed/
!wc -l data/processed/*.jsonl

## Chạy thử nhanh (~2 phút): 200 cặp, 4 bước
Kiểm tra môi trường trước khi chạy dài. Có thể bỏ qua.

In [ ]:
!python -X utf8 scripts/train_bartpho.py --limit 200 --val-limit 50 --max-steps 4 --batch-size 4 --output-dir outputs/checkpoints/bartpho_smoke
!python -X utf8 scripts/predict.py --model bartpho --checkpoint outputs/checkpoints/bartpho_smoke/best --data data/processed/vsec_test.jsonl --limit 20 --output /tmp/smoke.jsonl

## Train đầy đủ: 150k cặp, 3 epoch
Checkpoint tốt nhất theo loss Syn-val → `outputs/checkpoints/bartpho/best/`. Bị ngắt giữa chừng: thêm `--resume`.

In [ ]:
!python -X utf8 scripts/train_bartpho.py --config configs/bartpho.yaml     --train data/processed/syn_train.jsonl --val data/processed/syn_val.jsonl     --output-dir outputs/checkpoints/bartpho

## Predict 3 tập test (tag `bartpho`, beam 4)

In [ ]:
for name in ["syn_test", "vsec_test", "viwiki_test"]:
    !python -X utf8 scripts/predict.py --model bartpho --checkpoint outputs/checkpoints/bartpho/best         --data data/processed/{name}.jsonl --tag bartpho
!python -X utf8 scripts/evaluate.py --pred-dir outputs/predictions --out outputs/tables/main_results_bartpho.csv
!cat outputs/tables/main_results_bartpho.csv

## (Tùy chọn) Ablation mật độ nhiễu
3 mô hình × 50k cặp × 1 epoch (≈ 1550 bước mỗi mô hình, khoảng 25–35 phút mỗi mô hình trên T4), predict VSEC-test với tag `bartpho_d05/d12/d25`.

In [ ]:
if RUN_ABLATION:
    for d in ["d05", "d12", "d25"]:
        !python -X utf8 scripts/train_bartpho.py --train data/processed/syn_train_{d}.jsonl --limit 50000 --epochs 1             --eval-steps 500 --output-dir outputs/checkpoints/bartpho_{d}
        !python -X utf8 scripts/predict.py --model bartpho --checkpoint outputs/checkpoints/bartpho_{d}/best             --data data/processed/vsec_test.jsonl --tag bartpho_{d}
        !rm -rf outputs/checkpoints/bartpho_{d}/checkpoint-*   # tiết kiệm dung lượng /kaggle/working

## Nén dự đoán để tải về
Tải `bartpho_predictions.zip` (Kaggle: tab *Output*; Colab: ô dưới tự tải), giải nén vào `outputs/predictions/` trên máy.
Checkpoint (~1,6 GB) giữ trên Kaggle/Drive, không đưa vào git.

In [ ]:
!cd outputs/predictions && zip -j {WORK}/bartpho_predictions.zip bartpho*__*.jsonl
!cp outputs/checkpoints/bartpho/best/vspell_config.json {WORK}/bartpho_vspell_config.json
!pip freeze | grep -iE "^(torch|transformers|datasets|accelerate|sentencepiece)==" > {WORK}/bartpho_pip_freeze.txt
!ls -la {WORK}
try:
    from google.colab import files; files.download(f"{WORK}/bartpho_predictions.zip")
except ImportError:
    pass